# C2 · Setup Colab

Đặt source `C2_Base` trong Drive và chỉnh `REPO` nếu cần. `C2_work` lưu manifests, run logs, checkpoints và evaluation reports. Chọn GPU runtime để kiểm tra CUDA. Notebook in loại GPU runtime tự cấp.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
REPO = DRIVE / "C2_Base"
WORK = DRIVE / "C2_work"
DATA_ROOT = Path("/content/digiface72")  # Extracted, authorized 72-capture partition
MANIFEST = WORK / "manifests/digiface72.csv"

assert (REPO / "configs/pilot.yaml").is_file(), f"Set REPO to the copied C2_Base source folder: {REPO}"
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")

def run(module, *args):
    subprocess.run([sys.executable, "-m", module, *map(str, args)], cwd=REPO, check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")],
               cwd=REPO, check=True)

In [ ]:
import platform
import numpy as np
import scipy
import torch

info = {
    "python": platform.python_version(), "torch": str(torch.__version__),
    "numpy": np.__version__, "scipy": scipy.__version__,
    "cuda_available": torch.cuda.is_available(), "torch_cuda": torch.version.cuda,
}
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    info.update(gpu=properties.name, vram_bytes=properties.total_memory,
                bf16=torch.cuda.is_bf16_supported())
WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps(info, indent=2))
(WORK / "colab_runtime.json").write_text(json.dumps(info, indent=2), encoding="utf-8")

Chạy mathematical suite trước khi train. Nếu chọn mixed precision ở notebook training, chạy test CUDA cho đúng dtype trong runtime đó.

In [ ]:
run("pytest", "-q", "tests/test_math.py", "tests/test_models.py", "-m", "not cuda")
if torch.cuda.is_available():
    run("pytest", "-q", "-rs", "tests/test_models.py", "-m", "cuda")